# ⚡ Kilat: Supervised Fine-Tuning (SFT / Instruction Tuning) in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Airukua/Kilat/blob/main/notebooks/kilat_sft_colab.ipynb)

```
  ██╗  ██╗██╗██╗      █████╗ ████████╗
  ██║ ██╔╝██║██║     ██╔══██╗╚══██╔══╝
  █████╔╝ ██║██║     ███████║   ██║   
  ██╔═██╗ ██║██║     ██╔══██║   ██║   
  ██║  ██╗██║███████╗██║  ██║   ██║   
  ╚═╝  ╚═╝╚═╝╚══════╝╚═╝  ╚═╝   ╚═╝   
  ────────────────────────────────────────────────────────────
  ⚡ KILAT | Kernelized Lightweight Attention Transformer (v2.0.0)
  Architecture: Hybrid Sub-Quadratic Global Decay + Latent MLA
  ────────────────────────────────────────────────────────────
```

This tutorial guides you through **Supervised Fine-Tuning (SFT / Instruction Tuning)** on a pre-trained Kilat language model using the native **Kilat Python API** on Google Colab GPU accelerators.

### Core SFT Invariants in Kilat:
- **Selective Prompt Masking**: Prompt / instruction tokens are masked with `ignore_index = -100` so loss and backprop are strictly computed on the assistant / response tokens.
- **Causal Alignment**: `KilatTransformer.forward` performs single-token causal target shifting (`logits[:, :-1]` vs `labels[:, 1:]`).
- **Flexible Schemas**: Native support for **Alpaca** (`instruction`, `input`, `output`), **ChatML** (`messages`), and direct **Prompt-Response** pairs.

## 1. Hardware & Accelerator Verification
Ensure your Google Colab runtime is set to **GPU** (*Runtime -> Change runtime type -> T4 / A100 / L4 GPU*).

In [ ]:
# ==============================================================================
#   ██╗  ██╗██╗██╗      █████╗ ████████╗
#   ██║ ██╔╝██║██║     ██╔══██╗╚══██╔══╝
#   █████╔╝ ██║██║     ███████║   ██║   
#   ██╔═██╗ ██║██║     ██╔══██║   ██║   
#   ██║  ██╗██║███████╗██║  ██║   ██║   
#   ╚═╝  ╚═╝╚═╝╚══════╝╚═╝  ╚═╝   ╚═╝   
#   ────────────────────────────────────────────────────────────
#   ⚡ KILAT | Supervised Fine-Tuning (SFT / Instruction Tuning)
#   ────────────────────────────────────────────────────────────
# ==============================================================================
# Check NVIDIA GPU acceleration details
!nvidia-smi

## 2. Clone Repository & Install Kilat

In [ ]:
# ==============================================================================
# ⚡ KILAT: Installation & Dependencies
# ==============================================================================
!git clone https://github.com/Airukua/Kilat.git
%cd Kilat

# Install Kilat in editable mode along with production ML libraries
!pip install -e .
!pip install datasets transformers triton safetensors accelerate

## 3. Load Pre-Trained Model & Tokenizer
We load the official pre-trained 35M model from Hugging Face Hub: `AiRukua/tiny-stories-kilat-35M`.

In [ ]:
# ==============================================================================
# ⚡ KILAT: Pre-Trained Model & Tokenizer Loading
# ==============================================================================
import torch
from transformers import AutoTokenizer
from kilat.models import KilatTransformer

MODEL_ID = "AiRukua/tiny-stories-kilat-35M"
print(f"[KILAT] Downloading & loading base model: {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = KilatTransformer.from_pretrained(MODEL_ID)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"[KILAT] Model successfully loaded!")
print(f"[KILAT] Total Parameters:     {total_params:,}")
print(f"[KILAT] Trainable Parameters: {trainable_params:,}")

## 4. Prepare Instruction Dataset (`SFTDataset` & `SFTDataCollator`)
Kilat provides `SFTDataset` and `SFTDataCollator` to handle instruction-response formatting and selective prompt masking.

Here we create sample instruction-following data (you can also pass any JSONL file or Hugging Face dataset):

In [ ]:
# ==============================================================================
# ⚡ KILAT: Instruction Dataset Preparation
# ==============================================================================
from kilat.data.sft import SFTDataset, SFTDataCollator

# Example instruction dataset (Alpaca format)
instruction_data = [
    {
        "instruction": "Tell a story about a brave knight and a friendly dragon.",
        "input": "",
        "output": "Once upon a time, Sir Arthur met a green dragon named Sparky. Instead of fighting, they shared apples and became best friends."
    },
    {
        "instruction": "What should you do before crossing the street?",
        "input": "",
        "output": "Always stop, look left, look right, and listen carefully before crossing the street safely."
    },
    {
        "instruction": "Write a cheerful morning greeting.",
        "input": "",
        "output": "Good morning! Wishing you a bright, happy, and productive day full of joy!"
    },
    {
        "instruction": "Describe a magical forest.",
        "input": "",
        "output": "The magical forest was glowing with soft blue fireflies. The ancient trees whispered gentle songs in the wind."
    }
] * 16  # Repeat for demonstration

# SFTDataset accepts in-memory lists, file paths (.jsonl / .json), or Hugging Face datasets
train_dataset = SFTDataset(instruction_data)

# SFTDataCollator guarantees prompt tokens receive -100 labels
collator = SFTDataCollator(
    tokenizer=tokenizer,
    max_length=256,
    ignore_index=-100,
    append_eos_token=True,
)

# Inspect a single collated sample to verify prompt loss masking
sample_batch = collator([instruction_data[0]])
print("[KILAT] Inspected sample batch tensor shapes:")
print(f"  input_ids shape:      {sample_batch['input_ids'].shape}")
print(f"  labels shape:         {sample_batch['labels'].shape}")
print(f"  Prompt masked (-100): {(sample_batch['labels'] == -100).sum().item()} tokens")
print(f"  Response active:      {(sample_batch['labels'] != -100).sum().item()} tokens")

## 5. Supervised Fine-Tuning with `KilatSFTTrainer`
Configure fine-tuning hyperparameters and launch training with `KilatSFTTrainer`.

Fine-tuning best practices:
- **Lower Learning Rate**: Use `1e-5` to `5e-5` to avoid catastrophic forgetting of pre-trained representations.
- **Cosine Schedule**: Decay learning rate smoothly with a short warmup ($5\% - 10\%$ of steps).

In [ ]:
# ==============================================================================
# ⚡ KILAT: KilatSFTTrainer Fine-Tuning Execution
# ==============================================================================
from kilat.engine import TrainingArguments
from kilat.engine.sft_trainer import KilatSFTTrainer

precision_mode = "bf16" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else ("fp16" if torch.cuda.is_available() else "fp32")

training_args = TrainingArguments(
    output_dir="./checkpoints_sft",
    training_mode="steps",
    max_steps=50,                     # Fast demo run (adjust to 200-1000 for full dataset)
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,    # Effective batch size: 4 * 2 = 8
    learning_rate=3e-5,               # SFT learning rate
    weight_decay=0.01,
    warmup_steps=10,
    scheduler_type="cosine",
    precision=precision_mode,
    logging_steps=5,
    save_steps=25,
    save_checkpoints=True,
    save_total_limit=2,
    seed=42,
)

sft_trainer = KilatSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    max_seq_length=256,
)

# Run SFT
train_state = sft_trainer.train()
print(f"\n[KILAT] SFT completed successfully at step {train_state.global_step}!")

## 6. Interactive Instruction Generation
Evaluate the model's ability to follow instructions using `TextGenerator`.

In [ ]:
# ==============================================================================
# ⚡ KILAT: Instruction Generation Test
# ==============================================================================
from kilat.inference.generator import TextGenerator

device = "cuda" if torch.cuda.is_available() else "cpu"
generator = TextGenerator(model=model, tokenizer=tokenizer, device=device)

test_instructions = [
    "Below is an instruction that describes a task.\n\n### Instruction:\nTell a story about a brave knight and a friendly dragon.\n\n### Response:\n",
    "Below is an instruction that describes a task.\n\n### Instruction:\nWrite a cheerful morning greeting.\n\n### Response:\n",
]

print("=" * 65)
print("⚡ KILAT SFT INFERENCE EVALUATION")
print("=" * 65)

for prompt in test_instructions:
    print(f"\n[PROMPT]:\n{prompt}")
    response = generator.generate(
        prompt=prompt,
        max_new_tokens=64,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.1,
    )
    print(f"[MODEL RESPONSE]:\n{response}\n" + "-" * 65)

## 7. Model Serialization & Hub Export
Save the fine-tuned model and tokenizer in standard Hugging Face `safetensors` format.

In [ ]:
# ==============================================================================
# ⚡ KILAT: Export Fine-Tuned Model
# ==============================================================================
import os

export_directory = "./exported_kilat_sft_model"
os.makedirs(export_directory, exist_ok=True)

# Save fine-tuned model weights
model.save_pretrained(export_directory, safe_serialization=True)
tokenizer.save_pretrained(export_directory)

print(f"[KILAT] SFT model successfully exported to: '{export_directory}'")
print(f"[KILAT] Export directory contents: {os.listdir(export_directory)}")

# ------------------------------------------------------------------------------
# (Optional) Upload to Hugging Face Hub:
# ------------------------------------------------------------------------------
# from huggingface_hub import login, HfApi
# login()  # Provide your Hugging Face write token
# api = HfApi()
# api.upload_folder(
#     folder_path=export_directory,
#     repo_id="your-username/kilat-35m-instruct",
#     repo_type="model",
# )
# print("[KILAT] Fine-tuned model uploaded to Hugging Face Hub!")